In [10]:
import torch
import pandas as pd
import numpy as np
import joblib
from datetime import timedelta

from train import build_model, load_config, get_single_file_df

In [8]:
config = load_config("config.yaml")
device = torch.device("cpu")

INPUT_WINDOW = config["data"]["input_window"]
FEATURE_COLS = config["data"]["feature_cols"]

# Stazioni
STATIONS = {
    1: {"lat": 41.90, "lon": 12.50},
    2: {"lat": 45.46, "lon": 9.18},
    3: {"lat": 40.85, "lon": 14.27},
    4: {"lat": 44.49, "lon": 11.34},
}

TARGET_DATE = "2025-12-23"
START_HOUR = 14
END_HOUR = 15

In [17]:
scaler = joblib.load("scaler_features.pkl")

df = pd.read_csv(
    "../../../data/events_dataset_scaled.csv",
    sep=";",
    parse_dates=["datetime"]
)

df = df.sort_values("datetime")

In [18]:
df

,Date,Time,barometer,temperature,wind_speed,rain_rate,datetime,station_id,hour,hour_sin,hour_cos,month,month_sin,month_cos
2376,01-01-2025,20:30:00,1.205329,-1.497040,-0.759755,0.182322,2025-01-01 20:30:00,2,20,-0.866025,0.500000,1,5.000000e-01,0.866025
2377,01-01-2025,20:40:00,1.206107,-1.521572,-0.744705,0.000000,2025-01-01 20:40:00,2,20,-0.866025,0.500000,1,5.000000e-01,0.866025
2378,01-01-2025,20:50:00,1.206885,-1.570636,-0.714605,0.000000,2025-01-01 20:50:00,2,20,-0.866025,0.500000,1,5.000000e-01,0.866025
2379,01-01-2025,21:00:00,1.207663,-1.602177,-0.692031,0.000000,2025-01-01 21:00:00,2,21,-0.707107,0.707107,1,5.000000e-01,0.866025
0,02-01-2025,02:40:00,1.100274,-1.602177,-1.120952,0.182322,2025-01-02 02:40:00,1,2,0.500000,0.866025,1,5.000000e-01,0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4808,29-12-2025,02:40:00,0.978878,-2.233003,-0.692031,0.000000,2025-12-29 02:40:00,2,2,0.500000,0.866025,12,-2.449294e-16,1.000000
4809,30-12-2025,04:30:00,0.829468,-2.138379,-0.759755,0.182322,2025-12-30 04:30:00,2,4,0.866025,0.500000,12,-2.449294e-16,1.000000
4810,30-12-2025,04:40:00,0.828690,-2.159407,-0.714605,0.000000,2025-12-30 04:40:00,2,4,0.866025,0.500000,12,-2.449294e-16,1.000000
4811,30-12-2025,04:50:00,0.827133,-2.194453,-0.654406,0.000000,2025-12-30 04:50:00,2,4,0.866025,0.500000,12,-2.449294e-16,1.000000


In [12]:
def prepare_input(df_station, reference_time):
    df_window = df_station[
        (df_station["datetime"] < reference_time) &
        (df_station["datetime"] >= reference_time - timedelta(hours=INPUT_WINDOW))
    ]

    if len(df_window) != INPUT_WINDOW:
        raise ValueError("Finestra temporale incompleta")

    X = df_window[FEATURE_COLS].values
    X = torch.tensor(X, dtype=torch.float32).unsqueeze(0)
    return X.to(device)

In [13]:
def run_inference_for_station(station_id):
    print(f"Inferenza stazione {station_id}")

    model = build_model(config, device)
    model.load_state_dict(
        torch.load(f"client_{station_id}_final_model.pth", map_location=device)
    )
    model.eval()

    df_station = df[df["station_id"] == station_id]

    predictions = []

    for hour in range(START_HOUR, END_HOUR):
        ts = pd.Timestamp(f"{TARGET_DATE} {hour:02d}:00:00")

        X = prepare_input(df_station, ts)

        with torch.no_grad():
            y_log = model(X).cpu().numpy().squeeze()

        # 🔥 inverse log1p
        rain = np.expm1(y_log)

        predictions.append({
            "WEATHER_STATION_ID": station_id,
            "ISO_8601_TIMESTAMP": ts.isoformat(),
            "LONGITUDE": STATIONS[station_id]["lon"],
            "LATITUDE": STATIONS[station_id]["lat"],
            "RAINRATE_VALUE": max(0.0, float(rain))
        })

    return predictions

In [14]:
all_predictions = []

for station_id in STATIONS.keys():
    preds = run_inference_for_station(station_id)
    all_predictions.extend(preds)

output_df = pd.DataFrame(all_predictions)

output_df.to_csv(
    "rain_predictions_2025-12-23_14-15UTC.csv",
    index=False
)

output_df

Inferenza stazione 1


ValueError: Finestra temporale incompleta

In [24]:
# %% [markdown]
# # Inferenza Transformer - Federated Learning
# 
# Notebook per eseguire inferenza su modelli Transformer addestrati in Federated Learning.
# Il dataset è **già preprocessato** (feature scalate, rain_rate in log-scale, feature cicliche).

# %% [markdown]
# ## 1. Import

# %%
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# %% [markdown]
# ## 2. Configurazione

# %%
CONFIG = {
    # Path
    'models_dir': Path('./models'),
    'data_path': Path('../../../data/events_dataset_scaled.csv'),
    'output_path': Path('./predictions_output.csv'),
    
    # Parametri modello
    'input_window': 24,
    'input_dim': 8,
    
    # Orario predizioni (UTC)
    'target_hour_start': 14,
    'target_hour_end': 15,
    
    # Stazioni meteo
    'stations': {
        1: {'id': 'STATION_001', 'lat': 45.4642, 'lon': 9.1900},
        2: {'id': 'STATION_002', 'lat': 41.9028, 'lon': 12.4964},
        3: {'id': 'STATION_003', 'lat': 40.8518, 'lon': 14.2681},
        4: {'id': 'STATION_004', 'lat': 43.7696, 'lon': 11.2558}
    }
}

PyTorch version: 2.9.1
CUDA available: False


In [27]:
for client_id, station_info in CONFIG['stations'].items():
    print(f"Client ID: {client_id}, Station Info: {station_info}")

Client ID: 1, Station Info: {'id': 'STATION_001', 'lat': 45.4642, 'lon': 9.19}
Client ID: 2, Station Info: {'id': 'STATION_002', 'lat': 41.9028, 'lon': 12.4964}
Client ID: 3, Station Info: {'id': 'STATION_003', 'lat': 40.8518, 'lon': 14.2681}
Client ID: 4, Station Info: {'id': 'STATION_004', 'lat': 43.7696, 'lon': 11.2558}


In [28]:


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Feature columns (ordine identico al training)
FEATURE_COLS = [
    "barometer", "temperature", "wind_speed", "rain_rate",
    "hour_sin", "hour_cos", "month_sin", "month_cos"
]

# %% [markdown]
# ## 3. Architettura Modello

# %%
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim=8, d_model=64, nhead=4, num_layers=2, 
                 dim_feedforward=128, dropout=0.1, output_dim=1):
        super().__init__()
        
        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc_out = nn.Linear(d_model, output_dim)
        
    def forward(self, src):
        src = self.input_projection(src)
        src = self.pos_encoder(src)
        output = self.transformer_encoder(src)
        output = output[:, -1, :]
        output = self.fc_out(output)
        return output

print("✓ Modello definito")

# %% [markdown]
# ## 4. Caricamento Dati

# %%
# Carica dataset (già preprocessato, separatore punto e virgola)
df = pd.read_csv(CONFIG['data_path'], sep=';')
print(f"Dataset caricato: {df.shape}")
print(f"Colonne: {df.columns.tolist()}")

# Converti datetime
if df['datetime'].dtype == 'object':
    df['datetime'] = pd.to_datetime(df['datetime'])

# Ordina per datetime
df = df.sort_values(['station_id', 'datetime']).reset_index(drop=True)

print(f"\nRange date: {df['datetime'].min()} - {df['datetime'].max()}")
print(f"Stazioni presenti: {sorted(df['station_id'].unique())}")
print(f"\nPrime righe:")
print(df.head())

# %% [markdown]
# ## 5. Definizione Target Datetimes

# %%
# Usa l'ultima data disponibile nel dataset
latest_date = df['datetime'].dt.date.max()
print(f"Ultima data nel dataset: {latest_date}")

# Crea i datetime target (14:00 e 15:00 UTC)
target_datetimes = [
    datetime.combine(latest_date, datetime.min.time()) + timedelta(hours=CONFIG['target_hour_start']),
    datetime.combine(latest_date, datetime.min.time()) + timedelta(hours=CONFIG['target_hour_end'])
]

print(f"\nDatetime target per inferenza:")
for dt in target_datetimes:
    print(f"  - {dt.strftime('%Y-%m-%d %H:%M:%S')}")

# %% [markdown]
# ## 6. Funzioni Inferenza

# %%
def prepare_input_window(df_station, target_dt, window_size=24):
    """
    Prepara finestra di input per inferenza.
    
    Args:
        df_station: DataFrame di una stazione (già filtrato)
        target_dt: datetime per cui fare predizione
        window_size: numero di timestep precedenti
    
    Returns:
        input_tensor: (1, window_size, 8) o None se dati insufficienti
    """
    # Filtra dati prima del target
    df_before = df_station[df_station['datetime'] < target_dt].copy()
    
    if len(df_before) < window_size:
        return None
    
    # Prendi ultimi window_size timestep
    window_data = df_before.iloc[-window_size:][FEATURE_COLS].values
    
    # Converti in tensor
    input_tensor = torch.FloatTensor(window_data).unsqueeze(0)  # (1, 24, 8)
    
    return input_tensor


def run_inference(model_path, df_station, target_datetimes):
    """
    Esegue inferenza per una stazione.
    
    Args:
        model_path: path al modello .pth
        df_station: DataFrame filtrato per la stazione
        target_datetimes: lista di datetime
    
    Returns:
        predictions: lista di valori predetti (scala reale)
    """
    # Carica modello
    model = TimeSeriesTransformer(
        input_dim=CONFIG['input_dim'],
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.1,
        output_dim=1
    )
    
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    
    predictions = []
    
    with torch.no_grad():
        for target_dt in target_datetimes:
            # Prepara input
            input_tensor = prepare_input_window(df_station, target_dt, CONFIG['input_window'])
            
            if input_tensor is None:
                predictions.append(np.nan)
                print(f"    ⚠ {target_dt.strftime('%H:%M')} - Dati insufficienti")
                continue
            
            # Inferenza
            input_tensor = input_tensor.to(DEVICE)
            output = model(input_tensor)
            
            # Da log-scale a scala reale
            pred_log = output.cpu().numpy()[0, 0]
            pred_real = np.expm1(pred_log)
            
            predictions.append(pred_real)
            print(f"    ✓ {target_dt.strftime('%H:%M')} - rain_rate: {pred_real:.4f} mm/h")
    
    return predictions

# %% [markdown]
# ## 7. Esecuzione Inferenza

# %%
results = []

print(f"\n{'='*70}")
print(f"INIZIO INFERENZA")
print(f"{'='*70}")

for client_id, station_info in CONFIG['stations'].items():
    print(f"\n[STAZIONE {client_id}] {station_info['id']}")
    print(f"-" * 70)
    
    # Path modello
    model_path = f"client_{client_id}_final_model.pth"

    if not Path(model_path).exists():
        print(f"  ❌ Modello non trovato: {model_path}")
        continue

    print(f"  ✓ Modello caricato: {model_path}")

    # Filtra dati per stazione
    df_station = df[df['station_id'] == client_id].copy()
    print(f"  ✓ Dati stazione: {len(df_station)} righe")
    
    if len(df_station) == 0:
        print(f"  ❌ Nessun dato per stazione {client_id}")
        continue
    
    # Esegui inferenza
    try:
        predictions = run_inference(model_path, df_station, target_datetimes)
        
        # Salva risultati
        for dt, pred in zip(target_datetimes, predictions):
            results.append({
                'WEATHER_STATION_ID': station_info['id'],
                'ISO_8601_TIMESTAMP': dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
                'LONGITUDE': station_info['lon'],
                'LATITUDE': station_info['lat'],
                'RAINRATE_VALUE': pred
            })
    
    except Exception as e:
        print(f"  ❌ Errore: {type(e).__name__}: {str(e)}")

print(f"\n{'='*70}")
print(f"INFERENZA COMPLETATA - {len(results)} predizioni generate")
print(f"{'='*70}")

# %% [markdown]
# ## 8. Salvataggio Risultati

# %%
if len(results) == 0:
    print("\n❌ ERRORE: Nessuna predizione generata!")
    print("\nPossibili cause:")
    print("  - Modelli non trovati")
    print("  - Dati insufficienti per le finestre temporali")
    print(f"\nVerifica directory modelli: {CONFIG['models_dir']}")
else:
    # Crea DataFrame
    df_results = pd.DataFrame(results)
    
    # Ordina
    df_results = df_results.sort_values(['WEATHER_STATION_ID', 'ISO_8601_TIMESTAMP'])
    
    # Salva CSV
    df_results.to_csv(CONFIG['output_path'], index=False)
    
    print(f"\n✓ CSV salvato: {CONFIG['output_path']}")
    print(f"\nAnteprima risultati:")
    print(df_results.to_string(index=False))
    
    print(f"\n{'='*70}")
    print(f"STATISTICHE")
    print(f"{'='*70}")
    print(f"Totale predizioni: {len(df_results)}")
    print(f"\nPer stazione:")
    print(df_results.groupby('WEATHER_STATION_ID')['RAINRATE_VALUE'].agg(['count', 'mean', 'min', 'max']))
    print(f"\nDistribuzione globale:")
    print(df_results['RAINRATE_VALUE'].describe())

# %%

Device: cpu
✓ Modello definito
Dataset caricato: (10206, 14)
Colonne: ['Date', 'Time', 'barometer', 'temperature', 'wind_speed', 'rain_rate', 'datetime', 'station_id', 'hour', 'hour_sin', 'hour_cos', 'month', 'month_sin', 'month_cos']

Range date: 2025-01-01 20:30:00 - 2025-12-30 05:00:00
Stazioni presenti: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Prime righe:
         Date      Time  barometer  temperature  wind_speed  rain_rate  \
0  02-01-2025  02:40:00   1.100274    -1.602177   -1.120952   0.182322   
1  02-01-2025  02:50:00   1.099496    -1.584655   -1.181152   0.000000   
2  02-01-2025  03:00:00   1.096383    -1.591664   -1.301551   0.000000   
3  02-01-2025  03:10:00   1.095605    -1.602177   -1.391850   0.000000   
4  03-01-2025  13:40:00   0.796784     0.227217    1.001081   0.693147   

             datetime  station_id  hour  hour_sin  hour_cos  month  month_sin  \
0 2025-01-02 02:40:00           1     2  0.500000  0.866025      1        0.5   
1 2025-01-02 02:5

In [30]:
# %% [markdown]
# # Inferenza Transformer - Federated Learning
# 
# Notebook per eseguire inferenza su modelli Transformer addestrati in Federated Learning.
# Il dataset è **già preprocessato** (feature scalate, rain_rate in log-scale, feature cicliche).

# %% [markdown]
# ## 1. Import

# %%
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# %% [markdown]
# ## 2. Configurazione

# %%
CONFIG = {
    # Path
    'models_dir': Path(''),
    'data_path': Path('../../../data/events_dataset_scaled.csv'),
    'output_path': Path('./predictions_output.csv'),
    
    # Parametri modello
    'input_window': 24,
    'input_dim': 8,
    
    # Orario predizioni (UTC)
    'target_hour_start': 14,
    'target_hour_end': 15,
    
    # Stazioni meteo
    'stations': {
        1: {'id': 'STATION_001', 'lat': 45.4642, 'lon': 9.1900},
        2: {'id': 'STATION_002', 'lat': 41.9028, 'lon': 12.4964},
        3: {'id': 'STATION_003', 'lat': 40.8518, 'lon': 14.2681},
        4: {'id': 'STATION_004', 'lat': 43.7696, 'lon': 11.2558}
    }
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Feature columns (ordine identico al training)
FEATURE_COLS = [
    "barometer", "temperature", "wind_speed", "rain_rate",
    "hour_sin", "hour_cos", "month_sin", "month_cos"
]

# %% [markdown]
# ## 3. Architettura Modello

# %%
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TimeSeriesTransformer(nn.Module):
    """Architettura IDENTICA a quella usata in training."""
    def __init__(self, input_dim=8, d_model=64, nhead=4, num_layers=2, 
                 dim_feedforward=128, dropout=0.1, output_dim=1):
        super().__init__()
        
        # Nome layer esattamente come in training
        self.input_linear = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout, max_len=500)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        
        # Output layer come in training: fc è un Sequential
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, output_dim)
        )
        
    def forward(self, src):
        src = self.input_linear(src)
        src = self.pos_encoder(src)
        output = self.encoder(src)
        output = output[:, -1, :]
        output = self.fc(output)
        return output

print("✓ Modello definito (architettura training)")

# %% [markdown]
# ## 4. Caricamento Dati

# %%
# Carica dataset (già preprocessato, separatore punto e virgola)
df = pd.read_csv(CONFIG['data_path'], sep=';')
print(f"Dataset caricato: {df.shape}")
print(f"Colonne: {df.columns.tolist()}")

# Converti datetime
if df['datetime'].dtype == 'object':
    df['datetime'] = pd.to_datetime(df['datetime'])

# Ordina per datetime
df = df.sort_values(['station_id', 'datetime']).reset_index(drop=True)

print(f"\nRange date: {df['datetime'].min()} - {df['datetime'].max()}")
print(f"Stazioni presenti: {sorted(df['station_id'].unique())}")
print(f"\nPrime righe:")
print(df.head())

# %% [markdown]
# ## 5. Definizione Target Datetimes

# %%
# Usa l'ultima data disponibile nel dataset
latest_date = df['datetime'].dt.date.max()
print(f"Ultima data nel dataset: {latest_date}")

# Crea i datetime target (14:00 e 15:00 UTC)
target_datetimes = [
    datetime.combine(latest_date, datetime.min.time()) + timedelta(hours=CONFIG['target_hour_start']),
    datetime.combine(latest_date, datetime.min.time()) + timedelta(hours=CONFIG['target_hour_end'])
]

print(f"\nDatetime target per inferenza:")
for dt in target_datetimes:
    print(f"  - {dt.strftime('%Y-%m-%d %H:%M:%S')}")

# %% [markdown]
# ## 6. Funzioni Inferenza

# %%
def prepare_input_window(df_station, target_dt, window_size=24):
    """
    Prepara finestra di input per inferenza.
    
    Args:
        df_station: DataFrame di una stazione (già filtrato)
        target_dt: datetime per cui fare predizione
        window_size: numero di timestep precedenti
    
    Returns:
        input_tensor: (1, window_size, 8) o None se dati insufficienti
    """
    # Filtra dati prima del target
    df_before = df_station[df_station['datetime'] < target_dt].copy()
    
    if len(df_before) < window_size:
        return None
    
    # Prendi ultimi window_size timestep
    window_data = df_before.iloc[-window_size:][FEATURE_COLS].values
    
    # Converti in tensor
    input_tensor = torch.FloatTensor(window_data).unsqueeze(0)  # (1, 24, 8)
    
    return input_tensor


def run_inference(model_path, df_station, target_datetimes):
    """
    Esegue inferenza per una stazione.
    
    Args:
        model_path: path al modello .pth
        df_station: DataFrame filtrato per la stazione
        target_datetimes: lista di datetime
    
    Returns:
        predictions: lista di valori predetti (scala reale)
    """
    # Carica modello
    model = TimeSeriesTransformer(
        input_dim=CONFIG['input_dim'],
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.1,
        output_dim=1
    )
    
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    
    predictions = []
    
    with torch.no_grad():
        for target_dt in target_datetimes:
            # Prepara input
            input_tensor = prepare_input_window(df_station, target_dt, CONFIG['input_window'])
            
            if input_tensor is None:
                predictions.append(np.nan)
                print(f"    ⚠ {target_dt.strftime('%H:%M')} - Dati insufficienti")
                continue
            
            # Inferenza
            input_tensor = input_tensor.to(DEVICE)
            output = model(input_tensor)
            
            # Da log-scale a scala reale
            pred_log = output.cpu().numpy()[0, 0]
            pred_real = np.expm1(pred_log)
            
            predictions.append(pred_real)
            print(f"    ✓ {target_dt.strftime('%H:%M')} - rain_rate: {pred_real:.4f} mm/h")
    
    return predictions

# %% [markdown]
# ## 7. Esecuzione Inferenza

# %%
results = []

print(f"\n{'='*70}")
print(f"INIZIO INFERENZA")
print(f"{'='*70}")

for client_id, station_info in CONFIG['stations'].items():
    print(f"\n[STAZIONE {client_id}] {station_info['id']}")
    print(f"-" * 70)
    
    # Path modello
    model_path = CONFIG['models_dir'] / f"client_{client_id}_final_model.pth"
    
    if not model_path.exists():
        print(f"  ❌ Modello non trovato: {model_path}")
        continue
    
    print(f"  ✓ Modello caricato: {model_path.name}")
    
    # Filtra dati per stazione
    df_station = df[df['station_id'] == client_id].copy()
    print(f"  ✓ Dati stazione: {len(df_station)} righe")
    
    if len(df_station) == 0:
        print(f"  ❌ Nessun dato per stazione {client_id}")
        continue
    
    # Esegui inferenza
    try:
        predictions = run_inference(model_path, df_station, target_datetimes)
        
        # Salva risultati
        for dt, pred in zip(target_datetimes, predictions):
            results.append({
                'WEATHER_STATION_ID': station_info['id'],
                'ISO_8601_TIMESTAMP': dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
                'LONGITUDE': station_info['lon'],
                'LATITUDE': station_info['lat'],
                'RAINRATE_VALUE': pred
            })
    
    except Exception as e:
        print(f"  ❌ Errore: {type(e).__name__}: {str(e)}")

print(f"\n{'='*70}")
print(f"INFERENZA COMPLETATA - {len(results)} predizioni generate")
print(f"{'='*70}")

# %% [markdown]
# ## 8. Salvataggio Risultati

# %%
if len(results) == 0:
    print("\n❌ ERRORE: Nessuna predizione generata!")
    print("\nPossibili cause:")
    print("  - Modelli non trovati")
    print("  - Dati insufficienti per le finestre temporali")
    print(f"\nVerifica directory modelli: {CONFIG['models_dir']}")
else:
    # Crea DataFrame
    df_results = pd.DataFrame(results)
    
    # Ordina
    df_results = df_results.sort_values(['WEATHER_STATION_ID', 'ISO_8601_TIMESTAMP'])
    
    # Salva CSV
    df_results.to_csv(CONFIG['output_path'], index=False)
    
    print(f"\n✓ CSV salvato: {CONFIG['output_path']}")
    print(f"\nAnteprima risultati:")
    print(df_results.to_string(index=False))
    
    print(f"\n{'='*70}")
    print(f"STATISTICHE")
    print(f"{'='*70}")
    print(f"Totale predizioni: {len(df_results)}")
    print(f"\nPer stazione:")
    print(df_results.groupby('WEATHER_STATION_ID')['RAINRATE_VALUE'].agg(['count', 'mean', 'min', 'max']))
    print(f"\nDistribuzione globale:")
    print(df_results['RAINRATE_VALUE'].describe())

# %%

PyTorch version: 2.9.1
CUDA available: False
Device: cpu
✓ Modello definito (architettura training)
Dataset caricato: (10206, 14)
Colonne: ['Date', 'Time', 'barometer', 'temperature', 'wind_speed', 'rain_rate', 'datetime', 'station_id', 'hour', 'hour_sin', 'hour_cos', 'month', 'month_sin', 'month_cos']

Range date: 2025-01-01 20:30:00 - 2025-12-30 05:00:00
Stazioni presenti: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Prime righe:
         Date      Time  barometer  temperature  wind_speed  rain_rate  \
0  02-01-2025  02:40:00   1.100274    -1.602177   -1.120952   0.182322   
1  02-01-2025  02:50:00   1.099496    -1.584655   -1.181152   0.000000   
2  02-01-2025  03:00:00   1.096383    -1.591664   -1.301551   0.000000   
3  02-01-2025  03:10:00   1.095605    -1.602177   -1.391850   0.000000   
4  03-01-2025  13:40:00   0.796784     0.227217    1.001081   0.693147   

             datetime  station_id  hour  hour_sin  hour_cos  month  month_sin  \
0 2025-01-02 02:40:00        

In [32]:
# %% [markdown]
# # Inferenza Transformer - Federated Learning
# 
# Notebook per eseguire inferenza su modelli Transformer addestrati in Federated Learning.
# Il dataset è **già preprocessato** (feature scalate, rain_rate in log-scale, feature cicliche).

# %% [markdown]
# ## 1. Import

# %%
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# %% [markdown]
# ## 2. Configurazione

# %%
CONFIG = {
    # Path
    'models_dir': Path(''),
    'data_path': Path('../../../data/events_dataset_scaled.csv'),
    'output_path': Path('./predictions_output.csv'),
    
    # Parametri modello (DA CONFIG.YAML)
    'input_window': 12,      # ← MODIFICATO da 24 a 12
    'output_window': 6,      # ← AGGIUNTO
    'input_dim': 8,
    'output_dim': 1,
    'd_model': 128,          # ← MODIFICATO da 64 a 128
    'nhead': 4,
    'num_layers': 4,         # ← MODIFICATO da 2 a 4
    'dropout': 0.25,         # ← MODIFICATO da 0.1 a 0.25
    'dim_feedforward': 256,  # ← AGGIUNTO esplicitamente
    
    # Orario predizioni (UTC)
    'target_hour_start': 14,
    'target_hour_end': 15,
    
    # Stazioni meteo
    'stations': {
        1: {'id': 'STATION_001', 'lat': 45.4642, 'lon': 9.1900},
        2: {'id': 'STATION_002', 'lat': 41.9028, 'lon': 12.4964},
        3: {'id': 'STATION_003', 'lat': 40.8518, 'lon': 14.2681},
        4: {'id': 'STATION_004', 'lat': 43.7696, 'lon': 11.2558}
    }
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Feature columns (ordine identico al training)
FEATURE_COLS = [
    "barometer", "temperature", "wind_speed", "rain_rate",
    "hour_sin", "hour_cos", "month_sin", "month_cos"
]

# %% [markdown]
# ## 3. Architettura Modello

# %%
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class myTransformer(nn.Module):
    """Architettura ESATTA da model.py del training."""
    def __init__(self, input_dim, output_dim, input_window, output_window,
                 d_model=128, nhead=4, num_layers=4, dropout=0.25):
        super().__init__()

        self.input_window = input_window
        self.output_window = output_window

        # Proietta le feature di input nello spazio d_model
        self.input_linear = nn.Linear(input_dim, d_model)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=256,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Mapper finale da T_in → T_out
        # Flatten: [B, T_in, d_model] → [B, T_in * d_model]
        # Poi MLP → [B, T_out * output_dim] → reshape [B, T_out, output_dim]
        self.fc = nn.Sequential(
            nn.Flatten(),  # [B, T_in * d_model]
            nn.Linear(input_window * d_model, output_window * output_dim)
        )
        self.output_dim = output_dim

    def forward(self, src):
        """
        src: [B, T_in, input_dim]
        returns: [B, T_out, output_dim]
        """
        x = self.input_linear(src)
        x = self.pos_encoder(x)
        x = self.encoder(x)  # [B, T_in, d_model]
        x = self.fc(x)       # [B, T_out * output_dim]
        x = x.view(-1, self.output_window, self.output_dim)  # [B, T_out, output_dim]
        return x

print("✓ Modello definito (architettura myTransformer)")

# %% [markdown]
# ## 4. Caricamento Dati

# %%
# Carica dataset (già preprocessato, separatore punto e virgola)
df = pd.read_csv(CONFIG['data_path'], sep=';')
print(f"Dataset caricato: {df.shape}")
print(f"Colonne: {df.columns.tolist()}")

# Converti datetime
if df['datetime'].dtype == 'object':
    df['datetime'] = pd.to_datetime(df['datetime'])

# Ordina per datetime
df = df.sort_values(['station_id', 'datetime']).reset_index(drop=True)

print(f"\nRange date: {df['datetime'].min()} - {df['datetime'].max()}")
print(f"Stazioni presenti: {sorted(df['station_id'].unique())}")
print(f"\nPrime righe:")
print(df.head())

# %% [markdown]
# ## 5. Definizione Target Datetimes

# %%
# Usa l'ultima data disponibile nel dataset
latest_date = df['datetime'].dt.date.max()
print(f"Ultima data nel dataset: {latest_date}")

# Crea i datetime target (14:00 e 15:00 UTC)
target_datetimes = [
    datetime.combine(latest_date, datetime.min.time()) + timedelta(hours=CONFIG['target_hour_start']),
    datetime.combine(latest_date, datetime.min.time()) + timedelta(hours=CONFIG['target_hour_end'])
]

print(f"\nDatetime target per inferenza:")
for dt in target_datetimes:
    print(f"  - {dt.strftime('%Y-%m-%d %H:%M:%S')}")

# %% [markdown]
# ## 6. Funzioni Inferenza

# %%
def prepare_input_window(df_station, target_dt, window_size=12):
    """
    Prepara finestra di input per inferenza.
    
    Args:
        df_station: DataFrame di una stazione (già filtrato)
        target_dt: datetime per cui fare predizione
        window_size: numero di timestep precedenti (12 per questo modello)
    
    Returns:
        input_tensor: (1, window_size, 8) o None se dati insufficienti
    """
    # Filtra dati prima del target
    df_before = df_station[df_station['datetime'] < target_dt].copy()
    
    if len(df_before) < window_size:
        return None
    
    # Prendi ultimi window_size timestep
    window_data = df_before.iloc[-window_size:][FEATURE_COLS].values
    
    # Converti in tensor
    input_tensor = torch.FloatTensor(window_data).unsqueeze(0)  # (1, 12, 8)
    
    return input_tensor


def run_inference(model_path, df_station, target_datetimes):
    """
    Esegue inferenza per una stazione.
    
    Il modello predice output_window=6 timestep futuri.
    Prendiamo solo il primo step della predizione.
    
    Args:
        model_path: path al modello .pth
        df_station: DataFrame filtrato per la stazione
        target_datetimes: lista di datetime
    
    Returns:
        predictions: lista di valori predetti (scala reale)
    """
    # Carica modello con parametri corretti
    model = myTransformer(
        input_dim=CONFIG['input_dim'],
        output_dim=CONFIG['output_dim'],
        input_window=CONFIG['input_window'],
        output_window=CONFIG['output_window'],
        d_model=CONFIG['d_model'],
        nhead=CONFIG['nhead'],
        num_layers=CONFIG['num_layers'],
        dropout=CONFIG['dropout']
    )
    
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    
    predictions = []
    
    with torch.no_grad():
        for target_dt in target_datetimes:
            # Prepara input
            input_tensor = prepare_input_window(df_station, target_dt, CONFIG['input_window'])
            
            if input_tensor is None:
                predictions.append(np.nan)
                print(f"    ⚠ {target_dt.strftime('%H:%M')} - Dati insufficienti")
                continue
            
            # Inferenza
            input_tensor = input_tensor.to(DEVICE)
            output = model(input_tensor)  # (1, 6, 1) - predice 6 timestep
            
            # Prendi solo il primo timestep della predizione
            pred_log = output[0, 0, 0].cpu().numpy()
            
            # Da log-scale a scala reale
            pred_real = np.expm1(pred_log)
            
            predictions.append(pred_real)
            print(f"    ✓ {target_dt.strftime('%H:%M')} - rain_rate: {pred_real:.4f} mm/h")
    
    return predictions

# %% [markdown]
# ## 7. Esecuzione Inferenza

# %%
results = []

print(f"\n{'='*70}")
print(f"INIZIO INFERENZA")
print(f"{'='*70}")

for client_id, station_info in CONFIG['stations'].items():
    print(f"\n[STAZIONE {client_id}] {station_info['id']}")
    print(f"-" * 70)
    
    # Path modello
    model_path = CONFIG['models_dir'] / f"client_{client_id}_final_model.pth"
    
    if not model_path.exists():
        print(f"  ❌ Modello non trovato: {model_path}")
        continue
    
    print(f"  ✓ Modello caricato: {model_path.name}")
    
    # Filtra dati per stazione
    df_station = df[df['station_id'] == client_id].copy()
    print(f"  ✓ Dati stazione: {len(df_station)} righe")
    
    if len(df_station) == 0:
        print(f"  ❌ Nessun dato per stazione {client_id}")
        continue
    
    # Esegui inferenza
    try:
        predictions = run_inference(model_path, df_station, target_datetimes)
        
        # Salva risultati
        for dt, pred in zip(target_datetimes, predictions):
            results.append({
                'WEATHER_STATION_ID': station_info['id'],
                'ISO_8601_TIMESTAMP': dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
                'LONGITUDE': station_info['lon'],
                'LATITUDE': station_info['lat'],
                'RAINRATE_VALUE': pred
            })
    
    except Exception as e:
        print(f"  ❌ Errore: {type(e).__name__}: {str(e)}")

print(f"\n{'='*70}")
print(f"INFERENZA COMPLETATA - {len(results)} predizioni generate")
print(f"{'='*70}")

# %% [markdown]
# ## 8. Salvataggio Risultati

# %%
if len(results) == 0:
    print("\n❌ ERRORE: Nessuna predizione generata!")
    print("\nPossibili cause:")
    print("  - Modelli non trovati")
    print("  - Dati insufficienti per le finestre temporali")
    print(f"\nVerifica directory modelli: {CONFIG['models_dir']}")
else:
    # Crea DataFrame
    df_results = pd.DataFrame(results)
    
    # Ordina
    df_results = df_results.sort_values(['WEATHER_STATION_ID', 'ISO_8601_TIMESTAMP'])
    
    # Salva CSV
    df_results.to_csv(CONFIG['output_path'], index=False)
    
    print(f"\n✓ CSV salvato: {CONFIG['output_path']}")
    print(f"\nAnteprima risultati:")
    print(df_results.to_string(index=False))
    
    print(f"\n{'='*70}")
    print(f"STATISTICHE")
    print(f"{'='*70}")
    print(f"Totale predizioni: {len(df_results)}")
    print(f"\nPer stazione:")
    print(df_results.groupby('WEATHER_STATION_ID')['RAINRATE_VALUE'].agg(['count', 'mean', 'min', 'max']))
    print(f"\nDistribuzione globale:")
    print(df_results['RAINRATE_VALUE'].describe())

# %%

PyTorch version: 2.9.1
CUDA available: False
Device: cpu
✓ Modello definito (architettura myTransformer)
Dataset caricato: (10206, 14)
Colonne: ['Date', 'Time', 'barometer', 'temperature', 'wind_speed', 'rain_rate', 'datetime', 'station_id', 'hour', 'hour_sin', 'hour_cos', 'month', 'month_sin', 'month_cos']

Range date: 2025-01-01 20:30:00 - 2025-12-30 05:00:00
Stazioni presenti: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Prime righe:
         Date      Time  barometer  temperature  wind_speed  rain_rate  \
0  02-01-2025  02:40:00   1.100274    -1.602177   -1.120952   0.182322   
1  02-01-2025  02:50:00   1.099496    -1.584655   -1.181152   0.000000   
2  02-01-2025  03:00:00   1.096383    -1.591664   -1.301551   0.000000   
3  02-01-2025  03:10:00   1.095605    -1.602177   -1.391850   0.000000   
4  03-01-2025  13:40:00   0.796784     0.227217    1.001081   0.693147   

             datetime  station_id  hour  hour_sin  hour_cos  month  month_sin  \
0 2025-01-02 02:40:00   

In [5]:
config = load_config("config.yaml")
device = torch.device(config["training"]["device"])

In [6]:
model_path = "client_1_final_model.pth"

In [ ]:
df = read_csv("../../../data/storage/23-12-2025.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../../../data/storage/events_dataset copy.csv'

In [ ]:
model = build_model(config, device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()
print(f"Modello caricato da {model_path}")

In [ ]:
input_window = config["data"]["input_window"]
feature_cols = config["data"]["feature_cols"]

In [ ]:
X_input = df[feature_cols].values[-input_window:]  # ultimi input_window righe
X_input = torch.tensor(X_input, dtype=torch.float32).unsqueeze(0)  # batch=1, shape=(1, seq_len, input_dim)
X_input = X_input.to(device)

In [ ]:
with torch.no_grad():
    y_pred = model(X_input)

print("Predizione:", y_pred.cpu().numpy())

In [35]:
# %% [markdown]
# # Inferenza Semplice - Un Solo Modello

# %% [markdown]
# ## 1. Import

# %%
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from datetime import datetime

# %% [markdown]
# ## 2. Configurazione

# %%
# Path
MODEL_PATH = Path('client_1_final_model.pth')
DATA_PATH = Path('../../../data/events_dataset_scaled.csv')
OUTPUT_CSV = Path('predictions.csv')

# Parametri (da config.yaml)
INPUT_WINDOW = 12
OUTPUT_WINDOW = 6
INPUT_DIM = 8
OUTPUT_DIM = 1
D_MODEL = 128
NHEAD = 4
NUM_LAYERS = 4
DROPOUT = 0.25

# Feature columns
FEATURE_COLS = [
    "barometer", "temperature", "wind_speed", "rain_rate",
    "hour_sin", "hour_cos", "month_sin", "month_cos"
]

# Target datetime
STATION_ID = 1
TARGET_DATETIME = datetime(2025, 12, 23, 14, 0, 0)  # 23 dicembre 2025, 14:00 UTC

# Info stazione (per CSV)
STATION_INFO = {
    'id': 'STATION_001',
    'lat': 45.4642,
    'lon': 9.1900
}

DEVICE = torch.device('cpu')
print(f"Device: {DEVICE}")
print(f"Target: {TARGET_DATETIME}")

# %% [markdown]
# ## 3. Definizione Modello

# %%
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class myTransformer(nn.Module):
    def __init__(self, input_dim, output_dim, input_window, output_window,
                 d_model=128, nhead=4, num_layers=4, dropout=0.25):
        super().__init__()

        self.input_window = input_window
        self.output_window = output_window

        self.input_linear = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=256,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_window * d_model, output_window * output_dim)
        )
        self.output_dim = output_dim

    def forward(self, src):
        x = self.input_linear(src)
        x = self.pos_encoder(x)
        x = self.encoder(x)
        x = self.fc(x)
        x = x.view(-1, self.output_window, self.output_dim)
        return x

print("✓ Modello definito")

# %% [markdown]
# ## 4. Carica Dati

# %%
# Carica dataset
df = pd.read_csv(DATA_PATH, sep=';')
df['datetime'] = pd.to_datetime(df['datetime'])

# Filtra per stazione
df_station = df[df['station_id'] == STATION_ID].copy()
df_station = df_station.sort_values('datetime').reset_index(drop=True)

print(f"Dati stazione {STATION_ID}: {len(df_station)} righe")
print(f"Range: {df_station['datetime'].min()} - {df_station['datetime'].max()}")

# %% [markdown]
# ## 5. Prepara Input (12 righe prima delle 14:00)

# %%
# Filtra dati prima delle 14:00
df_before = df_station[df_station['datetime'] < TARGET_DATETIME].copy()

print(f"\nRighe prima di {TARGET_DATETIME}: {len(df_before)}")

# Prendi ultime 12 righe
input_data = df_before.iloc[-INPUT_WINDOW:][FEATURE_COLS].values

print(f"Input shape: {input_data.shape}")
print(f"\nUltime 12 righe (input):")
print(df_before.iloc[-INPUT_WINDOW:][['datetime'] + FEATURE_COLS])

# Converti in tensor
input_tensor = torch.FloatTensor(input_data).unsqueeze(0)  # (1, 12, 8)
print(f"\nInput tensor shape: {input_tensor.shape}")

# %% [markdown]
# ## 6. Carica Modello

# %%
model = myTransformer(
    input_dim=INPUT_DIM,
    output_dim=OUTPUT_DIM,
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

print(f"✓ Modello caricato: {MODEL_PATH}")

# %% [markdown]
# ## 7. Inferenza

# %%
with torch.no_grad():
    input_tensor = input_tensor.to(DEVICE)
    output = model(input_tensor)  # (1, 6, 1)

print(f"Output shape: {output.shape}")
print(f"\nPredizioni (6 timestep futuri, in log-scale):")
print(output[0, :, 0])

# Prendi il primo timestep (predizione per le 14:00)
pred_log = output[0, 0, 0].cpu().numpy()
pred_real = np.expm1(pred_log)

print(f"\n{'='*60}")
print(f"PREDIZIONE PER {TARGET_DATETIME}")
print(f"{'='*60}")
print(f"Valore log-scale: {pred_log:.6f}")
print(f"Rain rate (mm/h): {pred_real:.6f}")
print(f"{'='*60}")

# %% [markdown]
# ## 8. Salva Risultato in CSV (append mode)

# %%
# Crea riga risultato
result_row = {
    'WEATHER_STATION_ID': STATION_INFO['id'],
    'ISO_8601_TIMESTAMP': TARGET_DATETIME.strftime('%Y-%m-%dT%H:%M:%SZ'),
    'LONGITUDE': STATION_INFO['lon'],
    'LATITUDE': STATION_INFO['lat'],
    'RAINRATE_VALUE': pred_real
}

# Carica CSV esistente o crea nuovo DataFrame
if OUTPUT_CSV.exists():
    df_predictions = pd.read_csv(OUTPUT_CSV)
    print(f"\n✓ CSV esistente caricato: {len(df_predictions)} righe")
else:
    df_predictions = pd.DataFrame(columns=['WEATHER_STATION_ID', 'ISO_8601_TIMESTAMP', 
                                            'LONGITUDE', 'LATITUDE', 'RAINRATE_VALUE'])
    print(f"\n✓ Creato nuovo CSV")

# Aggiungi nuova riga
df_predictions = pd.concat([df_predictions, pd.DataFrame([result_row])], ignore_index=True)

# Salva (sovrascrive il file con i dati aggiornati)
df_predictions.to_csv(OUTPUT_CSV, index=False)

print(f"✓ Risultato salvato in: {OUTPUT_CSV}")
print(f"Totale righe nel CSV: {len(df_predictions)}")

print(f"\nUltime righe del CSV:")
print(df_predictions.tail())

# %%

# %%

Device: cpu
Target: 2025-12-23 14:00:00
✓ Modello definito
Dati stazione 1: 2376 righe
Range: 2025-01-02 02:40:00 - 2025-12-24 23:00:00

Righe prima di 2025-12-23 14:00:00: 2227
Input shape: (12, 8)

Ultime 12 righe (input):
                datetime  barometer  temperature  wind_speed  rain_rate  \
2215 2025-12-16 22:40:00   0.844253     0.174648   -0.473807   0.587787   
2216 2025-12-16 22:50:00   0.848144     0.111566   -0.774805   0.336472   
2217 2025-12-16 23:00:00   0.848922     0.083529   -0.774805   0.470004   
2218 2025-12-16 23:10:00   0.848922     0.076520   -0.594206   0.182322   
2219 2025-12-16 23:20:00   0.849701     0.090538   -0.217959   0.182322   
2220 2025-12-16 23:30:00   0.849701     0.104557   -0.052411   0.000000   
2221 2025-12-16 23:40:00   0.852813     0.125584   -0.112610   0.000000   
2222 2025-12-16 23:50:00   0.853980     0.132593   -0.240534   0.000000   
2223 2025-12-17 03:40:00   0.878493     0.206190   -1.211252   0.182322   
2224 2025-12-17 03:50:00 

RuntimeError: Error(s) in loading state_dict for myTransformer:
	Missing key(s) in state_dict: "encoder.layers.2.self_attn.in_proj_weight", "encoder.layers.2.self_attn.in_proj_bias", "encoder.layers.2.self_attn.out_proj.weight", "encoder.layers.2.self_attn.out_proj.bias", "encoder.layers.2.linear1.weight", "encoder.layers.2.linear1.bias", "encoder.layers.2.linear2.weight", "encoder.layers.2.linear2.bias", "encoder.layers.2.norm1.weight", "encoder.layers.2.norm1.bias", "encoder.layers.2.norm2.weight", "encoder.layers.2.norm2.bias", "encoder.layers.3.self_attn.in_proj_weight", "encoder.layers.3.self_attn.in_proj_bias", "encoder.layers.3.self_attn.out_proj.weight", "encoder.layers.3.self_attn.out_proj.bias", "encoder.layers.3.linear1.weight", "encoder.layers.3.linear1.bias", "encoder.layers.3.linear2.weight", "encoder.layers.3.linear2.bias", "encoder.layers.3.norm1.weight", "encoder.layers.3.norm1.bias", "encoder.layers.3.norm2.weight", "encoder.layers.3.norm2.bias". 
	size mismatch for i